# A/B Backtest Settings Comparison

Loads the latest A/B pair by default and compares:
- Run settings (`config_a.json` vs `config_b.json`)
- Summary metrics (`comparison.json`)
- Equity curves with benchmark reference


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def _resolve_backtest_root() -> Path:
    candidates = [Path('eval_results/backtest'), Path('../eval_results/backtest')]
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]

def _is_valid_pair_dir(path: Path) -> bool:
    cmp = path / 'comparison.json'
    cfg_a = path / 'config_a.json'
    cfg_b = path / 'config_b.json'
    return path.is_dir() and cmp.exists() and cfg_a.exists() and cfg_b.exists()

BACKTEST_ROOT = _resolve_backtest_root()
PAIR_ROOT = BACKTEST_ROOT / 'ab_pairs'
pair_dirs = []
if PAIR_ROOT.exists():
    pair_dirs = sorted([d for d in PAIR_ROOT.iterdir() if _is_valid_pair_dir(d)], key=lambda d: d.stat().st_mtime, reverse=True)
assert pair_dirs, f'No valid A/B pairs found under: {PAIR_ROOT.resolve()}'

top3 = pair_dirs[:3]
display(pd.DataFrame([
    {'rank': i + 1, 'pair_dir': d.name, 'modified_at': pd.to_datetime(d.stat().st_mtime, unit='s')}
    for i, d in enumerate(top3)
]))


In [ ]:
# Defaults to latest pair. Uncomment to pin a pair directory.
PAIR_DIR = top3[0]
# PAIR_DIR = PAIR_ROOT / '20260301T033708Z_regime_on_vs_off_latestcfg_full'

cmp_payload = json.loads((PAIR_DIR / 'comparison.json').read_text())
cfg_a = json.loads((PAIR_DIR / 'config_a.json').read_text())
cfg_b = json.loads((PAIR_DIR / 'config_b.json').read_text())

label_a = cmp_payload.get('labels', {}).get('a', 'A')
label_b = cmp_payload.get('labels', {}).get('b', 'B')
run_a_dir = Path(cmp_payload.get('run_a', {}).get('dir', PAIR_DIR / 'run_a'))
run_b_dir = Path(cmp_payload.get('run_b', {}).get('dir', PAIR_DIR / 'run_b'))

print('Pair:', PAIR_DIR.name)
print('Window requested:', cmp_payload.get('requested_window'))
print('Window actual:', cmp_payload.get('actual_window'))
print('Run A:', label_a, run_a_dir)
print('Run B:', label_b, run_b_dir)


In [ ]:
def _config_diff(a: dict, b: dict) -> pd.DataFrame:
    keys = sorted(set(a.keys()) | set(b.keys()))
    rows = []
    for k in keys:
        va = a.get(k, None)
        vb = b.get(k, None)
        if va != vb:
            rows.append({'key': k, 'A_value': va, 'B_value': vb})
    if not rows:
        return pd.DataFrame(columns=['key', 'A_value', 'B_value'])
    return pd.DataFrame(rows).sort_values('key').reset_index(drop=True)

cfg_diff = _config_diff(cfg_a, cfg_b)
print(f'Changed settings count: {len(cfg_diff)}')
display(cfg_diff)


In [ ]:
summary_rows = []
for label, key in [(label_a, 'run_a'), (label_b, 'run_b')]:
    s = cmp_payload.get(key, {}).get('summary', {})
    summary_rows.append({'label': label, **s})
summary_df = pd.DataFrame(summary_rows).set_index('label')
display(summary_df)

delta_payload = cmp_payload.get('delta_b_minus_a', {})
delta_df = pd.DataFrame([delta_payload], index=[f'{label_b}_minus_{label_a}'])
display(delta_df)


In [ ]:
eq_a = pd.read_csv(run_a_dir / 'equity_curve.csv')
eq_b = pd.read_csv(run_b_dir / 'equity_curve.csv')
for eq in (eq_a, eq_b):
    eq['trade_date'] = pd.to_datetime(eq['trade_date'])

fig, ax = plt.subplots(figsize=(11, 4))
for label, eq in [(label_a, eq_a), (label_b, eq_b)]:
    e = eq.sort_values('trade_date').copy()
    e['norm_nav'] = e['nav'] / e['nav'].iloc[0]
    ax.plot(e['trade_date'], e['norm_nav'], label=f'{label} portfolio')
    if 'benchmark_return' in e.columns:
        e['benchmark_norm'] = (1.0 + e['benchmark_return'].fillna(0.0)).cumprod()
        ax.plot(e['trade_date'], e['benchmark_norm'], linestyle='--', alpha=0.8, label=f'{label} benchmark')
ax.set_title('A/B Normalized NAV and Benchmark Reference')
ax.set_xlabel('trade_date')
ax.set_ylabel('normalized nav')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


In [ ]:
def _regime_stats(run_dir: Path) -> dict:
    path = run_dir / 'rebalance_log.jsonl'
    if not path.exists():
        return {'rows': 0, 'regime_switches': 0, 'labels': {}}
    rows = 0
    switches = 0
    labels = {}
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows += 1
            obj = json.loads(line)
            reg = obj.get('regime', {})
            lab = str(reg.get('label', 'unknown'))
            labels[lab] = labels.get(lab, 0) + 1
            if bool(reg.get('switched', False)):
                switches += 1
    return {'rows': rows, 'regime_switches': switches, 'labels': labels}

reg_df = pd.DataFrame([
    {'label': label_a, **_regime_stats(run_a_dir)},
    {'label': label_b, **_regime_stats(run_b_dir)},
]).set_index('label')
display(reg_df)
